In [2]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px
from sklearn.preprocessing import MultiLabelBinarizer
from surprise import Dataset, Reader, SVD
from surprise.model_selection import train_test_split
from surprise import accuracy
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import joblib

# Read and Inspect our Data

In [3]:
movie_content = pd.read_parquet("../data/clean/movie_content_clean.parquet")
ratings = pd.read_parquet("../data/clean/ratings_clean.parquet")


In [4]:
movie_content.sample(15)

,movieId,title,genres,tags
71806,231857,Marighella - Retrato Falado do Guerrilheiro (2...,Documentary,
28675,131984,Off the Mark (1987),(no genres listed),
85317,284747,Chris Rock: Selective Outrage (2023),Comedy,Chris Rock Funny as hell
76549,253302,Budapest Heist (2020),Action|Comedy,
64909,211446,The Ghost Who Walks (2019),Crime,
65746,213349,Metamorphosis (2019),Horror|Thriller,haunting
70404,226584,Being a Human Person (2020),Documentary,
24363,121239,A Life in Dirty Movies (2013),Documentary,movie business pornography movie business porn...
82933,278252,This is GWAR (2021),Documentary,documentary
14255,73912,Genevieve (1953),Comedy,brighton car race london england vintage car


In [5]:
movie_content.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 87585 entries, 0 to 87584
Data columns (total 4 columns):
 #   Column   Non-Null Count  Dtype   
---  ------   --------------  -----   
 0   movieId  87585 non-null  int32   
 1   title    87585 non-null  object  
 2   genres   87585 non-null  category
 3   tags     87585 non-null  string  
dtypes: category(1), int32(1), object(1), string(1)
memory usage: 1.9+ MB


In [6]:
ratings.sample(15)

,userId,movieId,rating
28625340,179379,1441,3.0
25012930,157049,7323,3.0
29113238,182407,210861,4.0
6445127,40218,364,4.0
2411276,15345,999,3.0
20509927,128434,92259,4.5
21852260,136724,7367,3.0
10688060,66973,58559,3.0
6102492,38086,3439,3.0
7447831,46580,2762,2.5


In [7]:
ratings.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32000204 entries, 0 to 32000203
Data columns (total 3 columns):
 #   Column   Dtype  
---  ------   -----  
 0   userId   int32  
 1   movieId  int32  
 2   rating   float32
dtypes: float32(1), int32(2)
memory usage: 366.2 MB


# Data Preprocessing and TF-IDF on Tags

In [8]:
# each movie has multiple genres separated by '|', we need to split them into lists, we'll use MultiLabelBinarizer 
movie_content['genres_list'] = movie_content['genres'].str.split('|')

mlb = MultiLabelBinarizer()
genre_features = mlb.fit_transform(movie_content['genres_list'])

movie_content.sample(15)

,movieId,title,genres,tags,genres_list
53490,186169,Jurassic School (2017),Adventure|Children,cloning dinosaur lost pet middle school pet sc...,"[Adventure, Children]"
39646,156840,Jurassic Attack (2012),Action|Sci-Fi,creature feature,"[Action, Sci-Fi]"
75012,246282,Real Life (2004),Drama|Fantasy,,"[Drama, Fantasy]"
45626,169662,Beautiful Veera (1950),Comedy,cossack dance fight gipsy parent child relatio...,[Comedy]
4793,4898,Novocaine (2001),Comedy|Crime|Mystery|Thriller,dentist Steve Martin twist Steve Martin dentis...,"[Comedy, Crime, Mystery, Thriller]"
32078,139604,Electrical Girl (2001),Comedy,erotic magic quirky sex comedy softcore,[Comedy]
15818,83345,"Bread, Love and Dreams (Pane, amore e fantasia...",Comedy|Romance,,"[Comedy, Romance]"
2578,2670,Run Silent Run Deep (1958),War,submarine World War II submarine World War II ...,[War]
45268,168920,Postcard (2010),Drama|Romance|War,shinobu ōtake suicide,"[Drama, Romance, War]"
61144,202567,Let Me Die a Woman (1977),Documentary,,[Documentary]


In [9]:
movie_content.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 87585 entries, 0 to 87584
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype   
---  ------       --------------  -----   
 0   movieId      87585 non-null  int32   
 1   title        87585 non-null  object  
 2   genres       87585 non-null  category
 3   tags         87585 non-null  string  
 4   genres_list  87585 non-null  object  
dtypes: category(1), int32(1), object(2), string(1)
memory usage: 2.6+ MB


In [10]:
# Get feature importance using TF-IDF vectorizer on the 'tags' column. This will help us identify which tags are most relevant for each movie and can be used as features in our recommendation system.
tfidf = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=5,
    max_features=10000,
    stop_words='english')
    

tfidf_matrix = tfidf.fit_transform(movie_content['tags'])
tfidf_matrix

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 1531965 stored elements and shape (87585, 10000)>

# Cosine Similarity Between Movies

In [53]:
# we will get cosine similarity between the top rated 10,000 movies for memory efficiency 

top_10000_movie = ratings['movieId'].value_counts().head(10_000).index

movie_to_idx = {id: i for i, id in enumerate(movie_content['movieId'])}
target_indices = [movie_to_idx[m_id] for m_id in top_10000_movie]

tfidf_subset = tfidf_matrix[target_indices]  

movie_similarity = cosine_similarity(tfidf_subset)

movie_similarity_df = pd.DataFrame(movie_similarity, index=top_10000_movie, columns=top_10000_movie)

In [54]:
movie_similarity_df        # similarity matrix for the top 10,000 movies based on their tags

movieId,318,356,296,2571,593,260,2959,480,527,4993,...,844,100469,95443,108787,183437,4147,5580,59295,7219,3960
movieId,,,,,,,,,,,,,,,,,,,,,
318,1.000000,0.205942,0.070407,0.098732,0.120240,0.051230,0.342298,0.034133,0.151300,0.076915,...,0.0,0.000280,0.003005,0.009033,0.0,0.012241,0.0,0.011483,0.001094,0.083714
356,0.205942,1.000000,0.081312,0.021205,0.171425,0.081419,0.068672,0.100390,0.218873,0.122590,...,0.0,0.000926,0.002453,0.015485,0.0,0.023221,0.0,0.001672,0.001458,0.015084
296,0.070407,0.081312,1.000000,0.050180,0.068188,0.049721,0.142946,0.024910,0.039364,0.060433,...,0.0,0.000179,0.001739,0.014706,0.0,0.005873,0.0,0.008122,0.031973,0.002324
2571,0.098732,0.021205,0.050180,1.000000,0.016094,0.273785,0.171296,0.157184,0.083816,0.063912,...,0.0,0.022328,0.003798,0.002419,0.0,0.003432,0.0,0.028389,0.012187,0.000288
593,0.120240,0.171425,0.068188,0.016094,1.000000,0.033526,0.162884,0.074375,0.108586,0.065303,...,0.0,0.000000,0.003900,0.041616,0.0,0.013624,0.0,0.001188,0.003786,0.018201
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4147,0.012241,0.023221,0.005873,0.003432,0.013624,0.007115,0.002121,0.007694,0.013521,0.007592,...,0.0,0.000000,0.000000,0.399084,0.0,1.000000,0.0,0.000000,0.000000,0.000000
5580,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.000000,0.000000,0.000000,0.0,0.000000,1.0,0.000000,0.000000,0.000000
59295,0.011483,0.001672,0.008122,0.028389,0.001188,0.054431,0.007863,0.052033,0.001491,0.000719,...,0.0,0.000000,0.000000,0.000000,0.0,0.000000,0.0,1.000000,0.000000,0.000000


# SVD Model

In [13]:
# Evaluate different numbers of factors (Just a check to see how it affects RMSE)

'''ratings = pd.read_parquet('../data/raw_parquet/ratings.parquet')

sample_df = ratings.sample(n=1000000, random_state=42)

reader = Reader(rating_scale=(0.5, 5.0))
sample_data = Dataset.load_from_df(sample_df[['userId', 'movieId', 'rating']], reader)

train, test = train_test_split(sample_data, test_size=0.2)

for k in [50, 60, 70, 80]:
    algo = SVD(n_factors=k, random_state=42)
    algo.fit(train)
    predictions = algo.test(test)
    print(f"Factors: {k} | RMSE: {accuracy.rmse(predictions):.4f}")'''

'ratings = pd.read_parquet(\'../data/raw_parquet/ratings.parquet\')\n\nsample_df = ratings.sample(n=1000000, random_state=42)\n\nreader = Reader(rating_scale=(0.5, 5.0))\nsample_data = Dataset.load_from_df(sample_df[[\'userId\', \'movieId\', \'rating\']], reader)\n\ntrain, test = train_test_split(sample_data, test_size=0.2)\n\nfor k in [50, 60, 70, 80]:\n    algo = SVD(n_factors=k, random_state=42)\n    algo.fit(train)\n    predictions = algo.test(test)\n    print(f"Factors: {k} | RMSE: {accuracy.rmse(predictions):.4f}")'

## We will use factors= 50 since RMSE just gets worse as number of factors increase

In [55]:
# We need to filter the ratings data to include only popular movies and active users for better model performance.
# Only keep movies with at least 500 ratings (Targeting popular stuff)
movie_counts = ratings['movieId'].value_counts()
popular_movies = movie_counts[movie_counts >= 500].index
ratings_filtered = ratings[ratings['movieId'].isin(popular_movies)]

# Only keep users with at least 100 ratings (Targeting 'Expert' users)
user_counts = ratings_filtered['userId'].value_counts()
active_users = user_counts[user_counts >= 100].index
ratings_filtered = ratings_filtered[ratings_filtered['userId'].isin(active_users)]

print(f"New Filtered rows: {len(ratings_filtered)}")
print(f"Unique Users: {ratings_filtered['userId'].nunique()}")

New Filtered rows: 24136359
Unique Users: 78883


In [56]:
# prepare data for SVD model (Use the filtered ratings data to decrease file size and improve performance)

reader = Reader(rating_scale=(0.5, 5.0))

data_svd = Dataset.load_from_df(ratings_filtered[['userId', 'movieId', 'rating']], reader)       # apply the reader to the dataframe


In [57]:
# turn data into a sparse matrix for SVD model

trainset = data_svd.build_full_trainset()

svd = SVD(n_factors=30,     # number of hidden features
         random_state=42)

svd.fit(trainset)


## Look at top movies for a given latent feature

In [31]:
# check the top 10 movies for a specific factor (latent feature)

movies_df = pd.read_parquet('../data/raw_parquet/movies.parquet')

title_map = dict(zip(movies_df['movieId'].astype(int), movies_df['title']))

factor_index = 0 

movie_factors = svd.qi
factor_scores = movie_factors[:, factor_index]

results = []      
for i, score in enumerate(factor_scores):
    raw_id = int(trainset.to_raw_iid(i))
    title = title_map.get(raw_id, f"Unknown (ID: {raw_id})")
    results.append((title, score))

top_10 = sorted(results, key=lambda x: x[1], reverse=True)[:10]

print(f"--- Top 10 Movies for Factor {factor_index} ---")
for title, score in top_10:
    print(f"[{score:.4f}] {title}")

--- Top 10 Movies for Factor 0 ---
[0.8508] Blair Witch Project, The (1999)
[0.8115] Mulholland Drive (2001)
[0.8021] Brokeback Mountain (2005)
[0.7980] Eyes Wide Shut (1999)
[0.7634] Cook the Thief His Wife & Her Lover, The (1989)
[0.7312] Babe: Pig in the City (1998)
[0.7015] Fahrenheit 9/11 (2004)
[0.6955] Spider-Man 2 (2004)
[0.6912] Tree of Life, The (2011)
[0.6771] Dogville (2003)


## Look at all the latent features and how they categorize movies 

In [32]:
movies_df = pd.read_parquet('../data/raw_parquet/movies.parquet')

title_map = dict(zip(movies_df['movieId'], movies_df['title']))    # map the titles so we get names

print(title_map.get(500))

Mrs. Doubtfire (1993)


In [33]:
from sklearn.manifold import TSNE

# Visualize the Movie Universe in 2D using t-SNE

movie_vectors = svd.qi

n_to_plot = 3000        # number of movies to plot

tsne = TSNE(n_components=2, perplexity=40, random_state=42, init='pca')
projection = tsne.fit_transform(movie_vectors[:n_to_plot])


plot_data = []
for i in range(n_to_plot):
    raw_id = trainset.to_raw_iid(i)
    title = title_map.get(int(raw_id), f"ID: {raw_id}")
    plot_data.append({
        'x': projection[i, 0],
        'y': projection[i, 1],
        'title': title
    })

df_plot = pd.DataFrame(plot_data)

fig = px.scatter(df_plot, x='x', y='y', text='title',
                 title="SVD Movie Universe: 50 Hidden Factors Reduced to 2D (3000 Movies)", hover_name='title', template='plotly_dark', width=1500, height=700)

fig.update_traces(textposition='top center', mode='markers')
fig.show()

# Create Recommendation Function based on Content-Based Filtering (Really Useful for new Users/ Cold Start)

In [34]:
# Recommend based on content similarity (Attributes like Genres and Tags)

def get_content_recs(movie_id, top_n=10):
    if movie_id not in movie_similarity_df.index:          # see if movie is in the similarity matrix (if it's not in the top 10,000 movies, it won't be)
        return []

    scores = movie_similarity_df.loc[movie_id]
    
    return (
        scores.sort_values(ascending=False).iloc[1:top_n+1].index.tolist())


# Create Recommendation Function based on Collaborative Filtering (SVD)

In [35]:
# Recommend based on User Behavior and Rating Patterns (Collaborative Patterns using SVD Model)

def get_collaborative_recs(user_id, n=10):
    watched = ratings[ratings['userId'] == user_id]['movieId'].unique()       # movies the user has already rated 

    all_movies = movie_content['movieId'].unique()
    unseen = [m for m in all_movies if m not in watched]        

    predictions = [(movie_id, svd.predict(user_id, movie_id).est) for movie_id in unseen]       # predict ratings for unseen movies

    predictions.sort(key=lambda x: x[1], reverse=True)           

    return [movie_id for movie_id, _ in predictions[:n]]


# Create Recommendation Function based on SVD and Content-Based Filtering for Existing User (Hybrid Approach)

### Here we use SVD for prediction and use cosine similarity to boost our results and their accuracy

In [36]:
# Recommend based on Hybrid Logic (Blends Movie Content and Rating Patterns)

def get_hybrid_recs(user_id, liked_movie_id, top_n=10):
    svd_recs = get_collaborative_recs(user_id, n=20)
    content_recs = get_content_recs(liked_movie_id, top_n=20)

    combined = list(set(svd_recs) | set(content_recs))       

    scored = []
    for movie_id in combined:
        svd_score = svd.predict(user_id, movie_id).est        
        content_score = (
            movie_similarity_df.loc[liked_movie_id, movie_id]
            if movie_id in movie_similarity_df.columns                  
            else 0)

        final_score = 0.7 * svd_score + 0.3 * content_score
        scored.append((movie_id, final_score))                 

    scored.sort(key=lambda x: x[1], reverse=True)

    return [movie_id for movie_id, _ in scored[:top_n]]


### Map Movie IDs to titles for our recommendations

In [37]:
def movie_ids_to_titles(movie_ids):                   # to display movie titles from their IDs
    return (
        movie_content[movie_content['movieId'].isin(movie_ids)]
        [['movieId', 'title']]
        .drop_duplicates().reset_index(drop=True))


# Save things we need for streamlit (Compact Versions of each)

In [38]:
'''joblib.dump(movie_similarity_df, "../Models/movie_similarity_df.pkl")

joblib.dump(svd, "../Models/svd_model.pkl")'''

'joblib.dump(movie_similarity_df, "../Models/movie_similarity_df.pkl")\n\njoblib.dump(svd, "../Models/svd_model.pkl")'

In [58]:
print("Shrinking Similarity Matrix...")
top_n = 300             
movie_similarity_df_compact = {}

# We only keep the top 300 most similar movies for each movie, and we convert the similarity scores to float16 to save space.
for movie_id in movie_similarity_df.index:
    # Get top 300 matches, convert to float16 to save even more space
    top_matches = movie_similarity_df.loc[movie_id].sort_values(ascending=False).iloc[1:top_n+1]
    movie_similarity_df_compact[movie_id] = top_matches.astype(np.float16).to_dict()

joblib.dump(movie_similarity_df_compact, "../Models/movie_similarity_df.pkl", compress=9)   # re-save the similarity matrix after shrinking
print("Done! Similarity is now a tiny dictionary.")

print("Shrinking SVD...")

# Define a real, simple class for the trainset to fix PicklingError
class MinimalTrainset:
    def __init__(self, user_map, item_map, global_mean):
        self._raw2inner_id_users = user_map
        self._raw2inner_id_items = item_map
        self.global_mean = global_mean
        self._raw_ratings = [] # Keeping history empty to save space

# Create a brand new empty SVD object 
small_svd = SVD(n_factors=30)

# Converting to float32 (only the internal math matrices)
small_svd.pu = svd.pu.astype(np.float32)
small_svd.qi = svd.qi.astype(np.float32)
small_svd.bu = svd.bu.astype(np.float32)
small_svd.bi = svd.bi.astype(np.float32)

# Attach the MinimalTrainset (with only the mappings and global mean, no ratings history) to the new SVD object
small_svd.trainset = MinimalTrainset(
    svd.trainset._raw2inner_id_users, 
    svd.trainset._raw2inner_id_items, 
    svd.trainset.global_mean
)

joblib.dump(small_svd, "../Models/svd_model.pkl", compress=9)      # re-save the SVD model after shrinking
print("Done! SVD is now much lighter.")

Shrinking Similarity Matrix...
Done! Similarity is now a tiny dictionary.
Shrinking SVD...
Done! SVD is now much lighter.


# End of Notebook